In [2]:
# Cell 1: Setup — run this first every time
import subprocess, sys
try:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "xgboost==2.0.2", "scikit-learn", "pandas",
                    "numpy", "requests", "joblib", "matplotlib",
                    "seaborn", "shap"], check=True)
except subprocess.CalledProcessError as e:
    print(f"[WARN] pip install failed: {e}. Continuing; required packages may already be installed.")

import requests, time, json, joblib, shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from xgboost import XGBClassifier

# Mount Google Drive for model persistence (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped (not in Colab)')

BASE_URL   = "https://api.india.delta.exchange/v2"
SYMBOL     = "BTCUSD"
PRODUCT_ID = 27

# ── Restricted timeframe list ────────────────────────────────────────────────
TIMEFRAMES = ["5m", "15m", "30m", "1h", "2h"]

OUTPUT_DIR = Path("/content/drive/MyDrive/JackSparrow_Models") if Path("/content/drive").exists() else Path("./models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTRACT_VALUE_BTC = 0.001
TAKER_FEE          = 0.0005
TP_PCT             = 0.008
SL_PCT             = 0.004
SLIPPAGE_BPS       = 5

print("✅ Setup complete")
print(f"   Active timeframes: {TIMEFRAMES}")
print(f"   Output directory: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete
   Active timeframes: ['5m', '15m', '30m', '1h', '2h']
   Output directory: /content/drive/MyDrive/JackSparrow_Models


In [3]:
# Cell 2: Fetch all perpetual futures data streams with pagination
from tqdm.notebook import tqdm

MAX_CANDLES    = 2000
REQUEST_DELAY  = 0.15         # reduced fetch pause so 5m is less slow
MAX_PAGES      = 30           # safety guard to prevent infinite loops
MAX_RETRIES    = 3
TIMEOUT        = 20

TF_TO_SECONDS = {
    "5m":  300,
    "15m": 900,
    "30m": 1800,
    "1h":  3600,
    "2h":  7200,
}

TF_DAYS_BACK = {
    "5m":  45,
    "15m": 90,
    "30m": 120,
    "1h":  180,
    "2h":  270,
}


def fetch_candles_paginated(symbol_query: str, resolution: str,
                             start_ts: float, end_ts: float) -> list:
    assert resolution in TF_TO_SECONDS, (
        f"Invalid resolution '{resolution}'. Allowed: {list(TF_TO_SECONDS.keys())}"
    )

    all_candles = []
    current_start = int(start_ts)
    batch = 0

    pbar = tqdm(total=int(end_ts - start_ts), desc=f"{symbol_query}@{resolution}", unit="s", leave=True, ncols=100)
    
    while current_start < int(end_ts):
        batch += 1
        if batch > MAX_PAGES:
            print(f"  [WARN] {symbol_query}@{resolution} batch {batch} exceeds MAX_PAGES ({MAX_PAGES}); stopping")
            break

        params = {
            "symbol": symbol_query,
            "resolution": resolution,
            "start": current_start,
            "end": int(end_ts),
        }

        for attempt in range(1, MAX_RETRIES + 1):
            t0 = time.time()
            try:
                r = requests.get(f"{BASE_URL}/history/candles", params=params, timeout=TIMEOUT)
                elapsed = time.time() - t0
                if elapsed > 5.0:
                    print(f"  [WARN] {symbol_query}@{resolution} batch {batch} took {elapsed:.1f}s")

                r.raise_for_status()
                data = r.json()
                break
            except Exception as exc:
                print(f"  [ERROR] {symbol_query}@{resolution} batch {batch} attempt {attempt} error: {exc}")
                if attempt == MAX_RETRIES:
                    print(f"  [ERROR] {symbol_query}@{resolution} batch {batch} reached max retries; aborting")
                    pbar.close()
                    return all_candles
                time.sleep(1.0)
        else:
            pbar.close()
            return all_candles

        if not data.get("success") or not data.get("result"):
            print(f"  [INFO] {symbol_query}@{resolution} batch {batch} no more data (success={data.get('success')})")
            break

        chunk = data["result"]
        if len(chunk) == 0:
            print(f"  [INFO] {symbol_query}@{resolution} batch {batch} empty chunk; stopping")
            break

        all_candles.extend(chunk)
        print(f"  [{symbol_query}@{resolution}] batch {batch}: {len(chunk)} candles (total: {len(all_candles)})")

        if len(chunk) < MAX_CANDLES:
            break

        last_t = chunk[-1].get("time")
        if last_t is None or last_t <= current_start:
            print(f"  [WARN] {symbol_query}@{resolution} batch {batch} invalid last_t ({last_t}) or no progress; stopping")
            break

        current_start = last_t + 1
        progress = current_start - int(start_ts)
        pbar.update(min(progress - pbar.n, pbar.total - pbar.n))
        time.sleep(REQUEST_DELAY)
    
    pbar.close()
    return all_candles


def build_df_for_timeframe(resolution: str) -> pd.DataFrame:
    days_back = TF_DAYS_BACK[resolution]
    end_ts    = time.time()
    start_ts  = end_ts - (days_back * 86400)

    print(f"\n{'='*55}")
    print(f"Extracting {resolution} | {days_back} days | {SYMBOL}")
    print(f"{'='*55}")

    raw_ohlcv   = fetch_candles_paginated(SYMBOL,              resolution, start_ts, end_ts)
    raw_mark    = fetch_candles_paginated(f"MARK:{SYMBOL}",    resolution, start_ts, end_ts)
    raw_funding = fetch_candles_paginated(f"FUNDING:{SYMBOL}", resolution, start_ts, end_ts)
    raw_oi      = fetch_candles_paginated(f"OI:{SYMBOL}",      resolution, start_ts, end_ts)

    if len(raw_ohlcv) == 0:
        print(f"  [ERROR] {resolution} no main candles, skipping dataframe build")
        return pd.DataFrame()

    df_ohlcv = pd.DataFrame(raw_ohlcv)
    df_mark  = pd.DataFrame(raw_mark)[["time","close"]].rename(columns={"close":"mark_price"}) if raw_mark else pd.DataFrame(columns=["time","mark_price"])
    df_fund  = pd.DataFrame(raw_funding)[["time","close"]].rename(columns={"close":"funding_rate"}) if raw_funding else pd.DataFrame(columns=["time","funding_rate"])
    df_oi    = pd.DataFrame(raw_oi)[["time","close"]].rename(columns={"close":"open_interest"}) if raw_oi else pd.DataFrame(columns=["time","open_interest"])

    df = (df_ohlcv
          .merge(df_mark,  on="time", how="left")
          .merge(df_fund,  on="time", how="left")
          .merge(df_oi,    on="time", how="left"))

    df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.set_index("datetime").sort_index()
    df[["funding_rate", "open_interest", "mark_price"]] = df[["funding_rate", "open_interest", "mark_price"]].ffill()

    if df.empty:
        print(f"  [ERROR] {resolution} resulting merged df empty")
        return df

    print(f"\n✅ {resolution}: {len(df)} rows × {len(df.columns)} cols ({df.index[0].date()} → {df.index[-1].date()})")
    return df

raw_data = {}
fetch_summary = []

print("\n🔄 Starting data fetch for all timeframes...\n")
for tf in tqdm(TIMEFRAMES, desc="Timeframes", position=0, leave=True, ncols=100):
    tf_df = build_df_for_timeframe(tf)
    raw_data[tf] = tf_df
    fetch_summary.append({"timeframe": tf, "rows": len(tf_df), "empty": tf_df.empty})
    print(f"   {tf}: {len(tf_df):,} rows (df empty? {tf_df.empty})")

print("\n✅ All timeframes fetch attempt complete.")

print("\n--- Fetch summary (fail-safe) ---")
print(f"{'TF':<5} {'rows':>8} {'empty':>8} {'status':>10}")
for rec in fetch_summary:
    status = "OK" if (rec['rows'] > 0 and not rec['empty']) else "FAILED"
    print(f"{rec['timeframe']:<5} {rec['rows']:>8,} {str(rec['empty']):>8} {status:>10}")
print("--- End summary ---\n")


🔄 Starting data fetch for all timeframes...



Timeframes:   0%|                                                             | 0/5 [00:00<?, ?it/s]


Extracting 5m | 45 days | BTCUSD


BTCUSD@5m:   0%|                                                         | 0/3888000 [00:00<?, ?s/s]

  [BTCUSD@5m] batch 1: 4000 candles (total: 4000)
  [BTCUSD@5m] batch 2: 3999 candles (total: 7999)
  [BTCUSD@5m] batch 3: 3998 candles (total: 11997)
  [BTCUSD@5m] batch 4: 3997 candles (total: 15994)
  [BTCUSD@5m] batch 5: 3996 candles (total: 19990)
  [BTCUSD@5m] batch 6: 3995 candles (total: 23985)
  [BTCUSD@5m] batch 7: 3994 candles (total: 27979)
  [BTCUSD@5m] batch 8: 3993 candles (total: 31972)
  [BTCUSD@5m] batch 9: 3992 candles (total: 35964)
  [BTCUSD@5m] batch 10: 3991 candles (total: 39955)
  [BTCUSD@5m] batch 11: 3990 candles (total: 43945)
  [BTCUSD@5m] batch 12: 3989 candles (total: 47934)
  [BTCUSD@5m] batch 13: 3988 candles (total: 51922)
  [BTCUSD@5m] batch 14: 3987 candles (total: 55909)
  [BTCUSD@5m] batch 15: 3986 candles (total: 59895)
  [BTCUSD@5m] batch 16: 3985 candles (total: 63880)
  [BTCUSD@5m] batch 17: 3984 candles (total: 67864)
  [BTCUSD@5m] batch 18: 3983 candles (total: 71847)
  [BTCUSD@5m] batch 19: 3982 candles (total: 75829)
  [BTCUSD@5m] batch 20:

...

OI:BTCUSD@5m:   0%|                                                      | 0/3888000 [00:00<?, ?s/s]

  [OI:BTCUSD@5m] batch 1: 4000 candles (total: 4000)
  [OI:BTCUSD@5m] batch 2: 3999 candles (total: 7999)
  [OI:BTCUSD@5m] batch 3: 3998 candles (total: 11997)
  [OI:BTCUSD@5m] batch 4: 3997 candles (total: 15994)
  [OI:BTCUSD@5m] batch 5: 3996 candles (total: 19990)
  [OI:BTCUSD@5m] batch 6: 3995 candles (total: 23985)
  [OI:BTCUSD@5m] batch 7: 3994 candles (total: 27979)
  [OI:BTCUSD@5m] batch 8: 3993 candles (total: 31972)
  [OI:BTCUSD@5m] batch 9: 3992 candles (total: 35964)
  [OI:BTCUSD@5m] batch 10: 3991 candles (total: 39955)
  [OI:BTCUSD@5m] batch 11: 3990 candles (total: 43945)
  [OI:BTCUSD@5m] batch 12: 3989 candles (total: 47934)
  [OI:BTCUSD@5m] batch 13: 3988 candles (total: 51922)
  [OI:BTCUSD@5m] batch 14: 3987 candles (total: 55909)
  [OI:BTCUSD@5m] batch 15: 3986 candles (total: 59895)
  [OI:BTCUSD@5m] batch 16: 3985 candles (total: 63880)
  [OI:BTCUSD@5m] batch 17: 3984 candles (total: 67864)
  [OI:BTCUSD@5m] batch 18: 3983 candles (total: 71847)
  [OI:BTCUSD@5m] batc

: 

: 

: 

In [ ]:
# Cell 3: Compute features for all timeframes

def compute_all_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    c = df["close"]

    out["returns"]     = c.pct_change()
    out["log_returns"] = np.log(c / c.shift(1))
    out["volatility"]  = out["log_returns"].rolling(20).std()

    for p in [9, 21, 50]:
        out[f"ema_{p}"] = c.ewm(span=p, adjust=False).mean()
    out["ema_cross_9_21"]  = out["ema_9"]  - out["ema_21"]
    out["ema_cross_21_50"] = out["ema_21"] - out["ema_50"]

    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    out["macd"]        = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    out["macd_hist"]   = out["macd"] - out["macd_signal"]

    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    out["rsi_14"] = 100 - (100 / (1 + rs))

    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - c.shift()).abs(),
        (df["low"]  - c.shift()).abs()
    ], axis=1).max(axis=1)
    out["atr_14"]  = tr.rolling(14).mean()
    out["atr_pct"] = out["atr_14"] / c

    sma20 = c.rolling(20).mean()
    std20 = c.rolling(20).std()
    bb_upper = sma20 + 2 * std20
    bb_lower = sma20 - 2 * std20
    out["bb_width"] = (bb_upper - bb_lower) / sma20
    out["bb_pct"]   = (c - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)

    plus_dm  = (df["high"].diff()).clip(lower=0)
    minus_dm = (-df["low"].diff()).clip(lower=0)
    tr14     = tr.rolling(14).sum()
    out["adx_14"] = (
        100 * (plus_dm.rolling(14).sum() - minus_dm.rolling(14).sum()).abs()
        / tr14.replace(0, np.nan)
    ).rolling(14).mean()

    vol = df["volume"]
    out["vol_zscore"] = (vol - vol.rolling(50).mean()) / vol.rolling(50).std()
    out["vol_ratio"]  = vol / vol.rolling(20).mean()

    out["roc_10"] = c.pct_change(10)
    out["roc_20"] = c.pct_change(20)

    body = df["close"] - df["open"]
    rng  = (df["high"] - df["low"]).replace(0, np.nan)
    out["cdl_body_ratio"]     = body / rng
    out["cdl_upper_wick"]     = (df["high"] - df[["open","close"]].max(axis=1)) / rng
    out["cdl_lower_wick"]     = (df[["open","close"]].min(axis=1) - df["low"]) / rng
    out["cdl_body_direction"] = np.sign(body)

    roll20h = df["high"].rolling(20).max()
    roll20l = df["low"].rolling(20).min()
    roll_rng = (roll20h - roll20l).replace(0, np.nan)
    out["sr_range_position"] = (c - roll20l) / roll_rng
    out["sr_near_high"]      = (out["sr_range_position"] > 0.95).astype(float)
    out["sr_near_low"]       = (out["sr_range_position"] < 0.05).astype(float)
    out["sr_breakout_up"]    = (c > roll20h.shift(1)).astype(float)

    mark = df["mark_price"].fillna(c)
    fund = df["funding_rate"].fillna(0.0)
    oi   = df["open_interest"].ffill().fillna(0.0)

    out["basis"]          = mark - c
    out["basis_pct"]      = out["basis"] / c
    out["basis_zscore"]   = ((out["basis_pct"] - out["basis_pct"].rolling(48).mean())
                             / out["basis_pct"].rolling(48).std().replace(0, np.nan))
    out["basis_direction"] = np.sign(out["basis"])

    out["funding_rate"]    = fund
    out["funding_cumsum3"] = fund.rolling(3).sum()
    f_mean = fund.rolling(168).mean()
    f_std  = fund.rolling(168).std().replace(0, np.nan)
    out["funding_zscore"]  = (fund - f_mean) / f_std
    out["funding_spike"]   = (out["funding_zscore"].abs() > 2.0).astype(float)
    out["funding_sign"]    = np.sign(fund)

    out["oi_change_pct"]    = oi.pct_change()
    out["oi_change_5"]      = oi.pct_change(5)
    price_dir = np.sign(c.diff())
    oi_dir    = np.sign(oi.diff())
    out["oi_price_confirm"] = (price_dir == oi_dir).astype(float) * 2 - 1
    oi_mean = oi.rolling(336).mean()
    oi_std  = oi.rolling(336).std().replace(0, np.nan)
    out["oi_zscore"] = (oi - oi_mean) / oi_std

    out["long_squeeze_risk"] = (
        (out["funding_sign"] > 0) & (out["funding_spike"] > 0) &
        (out["oi_change_pct"] < -0.01)
    ).astype(float)
    out["short_squeeze_risk"] = (
        (out["funding_sign"] < 0) & (out["funding_spike"] > 0) &
        (out["oi_change_pct"] < -0.01)
    ).astype(float)

    out["ob_imbalance"] = 0.0

    return out.replace([np.inf, -np.inf], np.nan).fillna(0.0)

featured_data = {}
for tf in TIMEFRAMES:
    print(f"Computing features for {tf}...")
    featured_data[tf] = compute_all_features(raw_data[tf])
    print(f"  {tf}: {featured_data[tf].shape}")

print("\n✅ Feature engineering complete for all timeframes.")

In [ ]:
# Cell 4: Label generation
TF_LOOKAHEAD = {
    "5m":  16,
    "15m": 12,
    "30m": 10,
    "1h":  8,
    "2h":  6,
}

TF_BARS_PER_8H = {
    "5m":  96,
    "15m": 32,
    "30m": 16,
    "1h":  8,
    "2h":  4,
}


def make_futures_labels(df: pd.DataFrame, tf: str,
                         tp_pct=TP_PCT, sl_pct=SL_PCT, fee=TAKER_FEE,
                         include_funding=True) -> pd.Series:
    lookahead  = TF_LOOKAHEAD[tf]
    bars_per_8h = TF_BARS_PER_8H[tf]

    close = df["close"].values
    high  = df["high"].values
    low   = df["low"].values
    fund  = df["funding_rate"].fillna(0).values if include_funding else np.zeros(len(df))
    n     = len(df)
    labels = np.ones(n, dtype=int)

    net_tp = tp_pct - 2 * fee
    tp_level = 1 + tp_pct
    sl_level = 1 - sl_pct

    for i in range(n - lookahead):
        entry    = close[i]
        tp_price = entry * tp_level
        sl_price = entry * sl_level
        cum_funding = 0.0

        for k in range(1, lookahead + 1):
            cum_funding += abs(fund[i + k]) / bars_per_8h
            bar_h = high[i + k]
            bar_l = low[i + k]
            effective_net_tp = net_tp - cum_funding

            if bar_h >= tp_price:
                if effective_net_tp > 0:
                    labels[i] = 2
                break
            elif bar_l <= sl_price:
                labels[i] = 0
                break

    return pd.Series(labels, index=df.index, dtype=int)

labeled_data = {}
for tf in TIMEFRAMES:
    df = featured_data[tf].copy()
    df["label"] = make_futures_labels(df, tf)
    labeled_data[tf] = df

    counts = df["label"].value_counts()
    total  = len(df)
    print(f"\n{tf} label distribution ({total} samples):")
    print(f"  SELL (0): {counts.get(0,0):>6}  ({counts.get(0,0)/total*100:.1f}%)")
    print(f"  HOLD (1): {counts.get(1,0):>6}  ({counts.get(1,0)/total*100:.1f}%)")
    print(f"  BUY  (2): {counts.get(2,0):>6}  ({counts.get(2,0)/total*100:.1f}%)")
    action_pct = (counts.get(0,0) + counts.get(2,0)) / total * 100
    if action_pct < 30:
        print(f"  ⚠️  Only {action_pct:.1f}% actionable labels — consider widening TP/SL")
    else:
        print(f"  ✅ {action_pct:.1f}% actionable labels (SELL + BUY)")

In [ ]:
# Cell 5: Train unified model per timeframe
N_FOLDS = 5
EXCLUDE = {"open", "high", "low", "close", "volume", "time",
           "mark_price", "funding_rate", "open_interest", "label"}

training_results = {}

for tf in TIMEFRAMES:
    print(f"\n{'='*55}")
    print(f"Training unified model: {tf}")
    print(f"{'='*55}")

    df = labeled_data[tf]
    FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]
    print(f"  Features: {len(FEATURE_COLS)}")

    X = df[FEATURE_COLS].values
    y = df["label"].values
    valid_mask = ~np.any(np.isnan(X), axis=1)
    X, y = X[valid_mask], y[valid_mask]
    print(f"  Valid samples: {len(X):,}  (dropped {(~valid_mask).sum()} rows)")

    unique_labels = np.unique(y)
    if len(X) < (N_FOLDS + 1) or len(unique_labels) < 2:
        print(f"  ⚠️  Skipping training for {tf}: too few samples ({len(X)}) or label classes ({len(unique_labels)})")
        training_results[tf] = {
            "f1": 0.0,
            "n_features": len(FEATURE_COLS),
            "n_samples": len(X),
            "dir": None,
            "model": None,
            "scaler": None,
            "features": FEATURE_COLS,
        }
        continue

    n_splits = min(N_FOLDS, max(2, len(X) // 10))
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=TF_LOOKAHEAD[tf])

    n_classes = len(unique_labels)
    if n_classes > 2:
        objective = "multi:softprob"
        num_class = n_classes
        eval_metric = "mlogloss"
    else:
        objective = "binary:logistic"
        num_class = 1
        eval_metric = "logloss"

    model_params = dict(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective=objective,
        num_class=num_class,
        eval_metric=eval_metric,
        tree_method="hist",
        verbosity=0,
    )

    f1_scores = []
    fold_reports = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = RobustScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_val_s   = scaler.transform(X_val)

        model = XGBClassifier(**model_params)
        model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)

        preds = model.predict(X_val_s)
        f1 = f1_score(y_val, preds, average="macro", zero_division=0)
        f1_scores.append(f1)
        fold_reports.append(classification_report(y_val, preds,
                              target_names=["SELL","HOLD","BUY"] if n_classes > 2 else ["NEG","POS"],
                              zero_division=0, output_dict=True))

    mean_f1 = float(np.mean(f1_scores)) if f1_scores else 0.0
    print(f"  CV F1 (macro): {mean_f1:.4f}  folds: [{', '.join(f'{s:.3f}' for s in f1_scores)}]")

    final_scaler = RobustScaler()
    X_scaled     = final_scaler.fit_transform(X)
    final_model  = XGBClassifier(**model_params)
    final_model.fit(X_scaled, y)

    tf_dir = OUTPUT_DIR / f"jacksparrow_v6_BTCUSD_{tf}"
    tf_dir.mkdir(exist_ok=True, parents=True)
    joblib.dump(final_model,  tf_dir / f"entry_model_BTCUSD_{tf}.joblib")
    joblib.dump(final_scaler, tf_dir / f"entry_scaler_BTCUSD_{tf}.joblib")

    feat_schema = {"features": FEATURE_COLS, "n_features": len(FEATURE_COLS)}
    (tf_dir / f"features_BTCUSD_{tf}.json").write_text(json.dumps(feat_schema, indent=2))

    metadata = {
        "model_name":         f"jacksparrow_BTCUSD_{tf}",
        "symbol":             SYMBOL,
        "timeframe":          tf,
        "product_id":         PRODUCT_ID,
        "contract_type":      "perpetual_futures",
        "contract_value_btc": CONTRACT_VALUE_BTC,
        "model_version":      "v6",
        "feature_count":      len(FEATURE_COLS),
        "features":           FEATURE_COLS,
        "includes_perpetual": True,
        "unified_model":      True,
        "active_timeframes":  TIMEFRAMES,
        "label_config": {
            "tp_pct":    TP_PCT,
            "sl_pct":    SL_PCT,
            "lookahead": TF_LOOKAHEAD[tf],
            "fee_rate":  TAKER_FEE,
        },
        "cv_f1_macro":  round(mean_f1, 4),
        "trained_at":   pd.Timestamp.utcnow().isoformat(),
    }
    (tf_dir / f"metadata_BTCUSD_{tf}.json").write_text(json.dumps(metadata, indent=2))

    training_results[tf] = {
        "f1":        mean_f1,
        "n_features": len(FEATURE_COLS),
        "n_samples":  len(X),
        "dir":        str(tf_dir),
        "model":      final_model,
        "scaler":     final_scaler,
        "features":   FEATURE_COLS,
    }
    print(f"  ✅ Saved to: {tf_dir}")

print(f"\n{'='*55}")
print("Training complete — summary:")
for tf, r in training_results.items():
    print(f"  {tf}: F1={r['f1']:.4f} | features={r['n_features']} | samples={r['n_samples']:,}")

print("\n--- Local model artifact validation ---")
for tf, r in training_results.items():
    if not r.get('dir'):
        print(f"  {tf}: no output directory (model not trained)")
        continue
    d = Path(r['dir'])
    cx = {
        'model': d / f"entry_model_BTCUSD_{tf}.joblib",
        'scaler': d / f"entry_scaler_BTCUSD_{tf}.joblib",
        'features': d / f"features_BTCUSD_{tf}.json",
        'metadata': d / f"metadata_BTCUSD_{tf}.json",
    }
    ok = True
    for k, path in cx.items():
        if not path.exists():
            print(f"  {tf}: MISSING {k} file {path}")
            ok = False
    if ok:
        print(f"  {tf}: all artifacts present locally")
print("--- End model validation ---\n")

In [ ]:
# Cell 6: Integrated vectorized backtest
COST_PER_TRADE = TAKER_FEE + (SLIPPAGE_BPS / 10_000)
TF_FUNDING_INTERVAL_BARS = {"5m":96, "15m":32, "30m":16, "1h":8, "2h":4}
TF_BARS_PER_YEAR = {"5m":105120, "15m":35040, "30m":17520, "1h":8760, "2h":4380}


def run_backtest(df_feat: pd.DataFrame, model, scaler, feature_cols: list,
                 tf: str, holdout_frac: float = 0.20) -> dict:
    n_holdout = int(len(df_feat) * holdout_frac)
    if n_holdout < 1:
        print(f"  ⚠️  {tf}: skip backtest, holdout segment too small ({len(df_feat)} bars)")
        empty_df = df_feat.iloc[0:0].copy()
        return {
            "timeframe": tf,
            "sharpe": 0.0,
            "max_drawdown": 0.0,
            "total_return": 0.0,
            "bnh_return": 0.0,
            "n_trades": 0,
            "win_rate": 0.0,
            "total_trade_cost": 0.0,
            "total_funding_cost": 0.0,
            "n_holdout_bars": 0,
            "df": empty_df,
        }
    df_bt = df_feat.iloc[-n_holdout:].copy()

    X = df_bt[feature_cols].replace([np.inf, -np.inf], 0).fillna(0).values
    X_s = scaler.transform(X)
    raw_preds = model.predict(X_s)
    signal_map = {0: -1, 1: 0, 2: 1}
    ml_signal = pd.Series([signal_map[p] for p in raw_preds], index=df_bt.index, name="ml_signal")

    df_bt["ml_signal"] = ml_signal
    df_bt["position"] = df_bt["ml_signal"].shift(1).fillna(0)
    df_bt["market_return"] = df_bt["close"].pct_change()
    df_bt["strategy_return"] = df_bt["position"] * df_bt["market_return"]

    df_bt["signal_change"] = (df_bt["position"].diff().abs() > 0).astype(float)
    df_bt["trade_cost"] = df_bt["signal_change"] * COST_PER_TRADE * 2

    funding_interval = TF_FUNDING_INTERVAL_BARS[tf]
    funding_rate_col = "funding_rate" if "funding_rate" in df_bt.columns else None
    funding_costs = pd.Series(0.0, index=df_bt.index)
    if funding_rate_col:
        for i in range(0, len(df_bt), funding_interval):
            bar_idx = df_bt.index[i]
            if df_bt.loc[bar_idx, "position"] != 0:
                funding_costs.iloc[i] = abs(float(df_bt.loc[bar_idx, funding_rate_col]))
    df_bt["funding_cost"] = funding_costs

    sl_cost = pd.Series(0.0, index=df_bt.index)
    tp_bonus = pd.Series(0.0, index=df_bt.index)
    entry_price = None
    position_side = 0

    for i, (idx, row) in enumerate(df_bt.iterrows()):
        pos = row["position"]
        if position_side == 0 and pos != 0:
            entry_price = row["close"]
            position_side = pos
            continue

        if position_side != 0 and entry_price is not None:
            if position_side == 1:
                sl_hit = row["low"] <= entry_price * (1 - SL_PCT)
                tp_hit = row["high"] >= entry_price * (1 + TP_PCT)
            else:
                sl_hit = row["high"] >= entry_price * (1 + SL_PCT)
                tp_hit = row["low"] <= entry_price * (1 - TP_PCT)

            if sl_hit:
                sl_cost.iloc[i] = SL_PCT + COST_PER_TRADE * 2
                position_side = 0
                entry_price = None
            elif tp_hit:
                tp_bonus.iloc[i] = TP_PCT - COST_PER_TRADE * 2
                position_side = 0
                entry_price = None

        if pos != position_side and pos == 0:
            position_side = 0
            entry_price = None

    df_bt["sl_adjustment"] = -sl_cost
    df_bt["tp_adjustment"] = tp_bonus

    df_bt["net_return"] = (
        df_bt["strategy_return"]
        - df_bt["trade_cost"]
        - df_bt["funding_cost"]
        + df_bt["sl_adjustment"]
        + df_bt["tp_adjustment"]
    )

    df_bt["cum_market"] = (1 + df_bt["market_return"]).cumprod()
    df_bt["cum_strategy"] = (1 + df_bt["net_return"]).cumprod()

    peak = df_bt["cum_strategy"].cummax()
    df_bt["drawdown"] = (df_bt["cum_strategy"] - peak) / peak
    max_drawdown = float(df_bt["drawdown"].min())

    bars_per_year = TF_BARS_PER_YEAR[tf]
    net_r = df_bt["net_return"].dropna()
    sharpe = float((net_r.mean()/net_r.std())*np.sqrt(bars_per_year) if net_r.std()>0 else 0.0)

    n_trades = int(df_bt["signal_change"].sum())
    trade_returns = df_bt["net_return"][df_bt["signal_change"]==1]
    win_rate = float((trade_returns>0).mean()) if len(trade_returns)>0 else 0.0

    total_return = float(df_bt["cum_strategy"].iloc[-1] - 1) if len(df_bt)>0 else 0.0
    bnh_return = float(df_bt["cum_market"].iloc[-1] - 1) if len(df_bt)>0 else 0.0
    total_trade_cost = float(df_bt["trade_cost"].sum())
    total_funding = float(df_bt["funding_cost"].sum())

    return {
        "timeframe": tf,
        "sharpe": round(sharpe, 4),
        "max_drawdown": round(max_drawdown, 4),
        "total_return": round(total_return, 4),
        "bnh_return": round(bnh_return, 4),
        "n_trades": n_trades,
        "win_rate": round(win_rate, 4),
        "total_trade_cost": round(total_trade_cost, 4),
        "total_funding_cost": round(total_funding, 4),
        "n_holdout_bars": len(df_bt),
        "df": df_bt,
    }

backtest_results = {}
for tf in TIMEFRAMES:
    res = training_results.get(tf, {})
    if not res or res.get('model') is None or res.get('scaler') is None:
        print(f"\n{'='*55}")
        print(f"Skipping backtest for {tf}: no trained model available.")
        continue

    df = labeled_data[tf]

    print(f"\n{'='*55}")
    print(f"Backtesting: {tf}")
    bt = run_backtest(df_feat=df, model=res['model'], scaler=res['scaler'], feature_cols=res['features'], tf=tf)
    backtest_results[tf] = bt

    print(f"  Sharpe Ratio:        {bt['sharpe']:.4f}")
    print(f"  Max Drawdown:        {bt['max_drawdown']:.2%}")
    print(f"  Total Return:        {bt['total_return']:.2%}")
    print(f"  Buy & Hold Return:   {bt['bnh_return']:.2%}")
    print(f"  Win Rate:            {bt['win_rate']:.2%}")
    print(f"  Number of Trades:    {bt['n_trades']}")
    print(f"  Total Trade Cost:    {bt['total_trade_cost']:.4%}")
    print(f"  Total Funding Cost:  {bt['total_funding_cost']:.4%}")

    if bt['sharpe'] < 0.5:
        print('  ⚠️  Sharpe < 0.5 — consider adjusting TP/SL or feature set')
    elif bt['max_drawdown'] < -0.20:
        print('  ⚠️  MDD > 20% — risk parameters may be too aggressive')
    else:
        print('  ✅ Backtest metrics acceptable')

    tf_dir = Path(res['dir']) if res.get('dir') else None
    if tf_dir is not None:
        meta_path = tf_dir / f"metadata_BTCUSD_{tf}.json"
        if meta_path.exists():
            metadata = json.loads(meta_path.read_text())
            metadata['backtest'] = {
                'sharpe': bt['sharpe'],
                'max_drawdown': bt['max_drawdown'],
                'total_return': bt['total_return'],
                'bnh_return': bt['bnh_return'],
                'n_trades': bt['n_trades'],
                'win_rate': bt['win_rate'],
                'holdout_bars': bt['n_holdout_bars'],
                'total_trade_cost': bt['total_trade_cost'],
                'total_funding_cost': bt['total_funding_cost'],
            }
            meta_path.write_text(json.dumps(metadata, indent=2))
            print('  ✅ Backtest summary appended to metadata')

fig, axes = plt.subplots(len(TIMEFRAMES), 1, figsize=(14, 4*len(TIMEFRAMES)))
if len(TIMEFRAMES) == 1:
    axes = [axes]

for ax, tf in zip(axes, TIMEFRAMES):
    bt = backtest_results.get(tf)
    if bt is None:
        continue
    df_bt = bt['df']
    if df_bt.empty:
        continue
    ax.plot(df_bt.index, df_bt['cum_market'], label='Buy & Hold', alpha=0.6, lw=1.2)
    ax.plot(df_bt.index, df_bt['cum_strategy'], label='ML Strategy', lw=1.5)
    ax.fill_between(df_bt.index, df_bt['drawdown'] + 1, 1, alpha=0.15, color='red', label='Drawdown')
    ax.set_title(f"{tf}  |  Sharpe: {bt['sharpe']:.2f}  MDD: {bt['max_drawdown']:.1%}  Return: {bt['total_return']:.1%}  Trades: {bt['n_trades']}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel('Cumulative Return')

plt.tight_layout()
plot_path = OUTPUT_DIR / 'backtest_equity_curves.png'
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"\n✅ Equity curves saved: {plot_path}")

In [ ]:
# Cell 7: SHAP feature importance analysis
for tf in TIMEFRAMES:
    res = training_results.get(tf, {})
    if not res or res.get('model') is None or res.get('scaler') is None:
        print(f"{tf}: no trained model to run SHAP")
        continue

    df = labeled_data[tf]
    FEATURE_COLS = res['features']
    n_classes = len(np.unique(df['label']))
    X = df[FEATURE_COLS].replace([np.inf, -np.inf], 0).fillna(0).values
    X_s = res['scaler'].transform(X)

    sample_size = min(500, len(X_s))
    if sample_size <= 0:
        print(f"{tf}: no samples for SHAP")
        continue

    sample_idx = np.random.choice(len(X_s), sample_size, replace=False)
    X_sample = X_s[sample_idx]

    try:
        explainer = shap.TreeExplainer(res['model'])
        shap_values = explainer.shap_values(X_sample)
    except Exception as e:
        print(f"  ⚠️  SHAP failed for {tf}: {e}")
        continue

    if isinstance(shap_values, list):
        if len(shap_values) >= n_classes and n_classes >= 3:
            # multiclass: pick two classes for comparison
            mean_abs_shap = np.mean([np.abs(shap_values[0]), np.abs(shap_values[-1])], axis=0)
        else:
            mean_abs_shap = np.mean([np.abs(arr) for arr in shap_values], axis=0)
    else:
        mean_abs_shap = np.abs(shap_values)

    if mean_abs_shap.ndim > 1:
        mean_abs_shap = np.mean(mean_abs_shap, axis=0)

    top_idx = np.argsort(mean_abs_shap)[-20:][::-1]
    top_features = [(FEATURE_COLS[i], mean_abs_shap[i]) for i in top_idx]

    print(f"\n{tf} — Top 20 Features (mean |SHAP|):")
    for feat, importance in top_features:
        bar = "█" * int(importance / (max(mean_abs_shap) + 1e-12) * 30)
        print(f"  {feat:<28} {importance:.5f}  {bar}")

print("\n✅ SHAP analysis complete.")

In [ ]:
# Cell 8: Final validation summary
print("\n" + "="*55)
print("FINAL VALIDATION SUMMARY")
print("="*55)

all_pass = True

for tf in TIMEFRAMES:
    tf_dir = OUTPUT_DIR / f"jacksparrow_v6_BTCUSD_{tf}"
    required_files = [
        f"entry_model_BTCUSD_{tf}.joblib",
        f"entry_scaler_BTCUSD_{tf}.joblib",
        f"features_BTCUSD_{tf}.json",
        f"metadata_BTCUSD_{tf}.json",
    ]
    missing = [f for f in required_files if not (tf_dir / f).exists()]

    if missing:
        print(f"  ❌ {tf}: MISSING {missing}")
        all_pass = False
        continue

    metadata = json.loads((tf_dir / f"metadata_BTCUSD_{tf}.json").read_text())
    bt = metadata.get('backtest', {})

    print(f"\n  {tf}:")
    print(f"    ✅ All 4 artifacts present")
    print(f"    includes_perpetual: {metadata.get('includes_perpetual')}")
    print(f"    unified_model:      {metadata.get('unified_model')}")
    print(f"    features:           {metadata.get('feature_count')}")
    print(f"    cv_f1_macro:        {metadata.get('cv_f1_macro'):.4f}")
    print(f"    backtest.sharpe:    {bt.get('sharpe', 'N/A')}")
    print(f"    backtest.mdd:       {bt.get('max_drawdown', 'N/A')}")
    print(f"    backtest.trades:    {bt.get('n_trades', 'N/A')}")

    if not metadata.get('includes_perpetual'):
        print(f"    ⚠️  includes_perpetual not set!")
        all_pass = False
    if bt.get('sharpe', 0) < 0:
        print(f"    ⚠️  Negative Sharpe — review model or label quality")

print(f"\n{'='*55}")
if all_pass:
    print("✅ ALL TIMEFRAMES VALIDATED — ready for deployment")
    print(f"   Output: {OUTPUT_DIR}")
else:
    print("❌ VALIDATION FAILED — check errors above before deploying")

print(f"\nActive timeframes: {TIMEFRAMES}")
print("Next step: Update MODEL_DIR in .env to point at this output directory.")
print("Reminder: Set leverage to 5x + Isolated Margin manually in Delta Exchange UI.")
print("Reminder: Enable 2FA on your Delta Exchange account before activating the bot.")